# AMEX Enterprise Credit Risk Platform
## Notebook 46 -- Early Warning System v2: Enhanced Modeling
### Phase 3 . Problem Statement 7: Early Warning System (Enhancement)

CRISP-DM stage: **Modeling (methodological improvement pass)**. Depends on Problem 1 Notebooks 01/02/05 and Problem 7's own Notebooks 42/43/44 (the v1 policy and honest, NOT-RECOMMENDED v1 result).

**Why this notebook exists:** v1 (Notebooks 42-45) correctly reported that its winning candidate did not clear the platform's own >=1.5x default-rate-lift KPI, and was honestly flagged NOT RECOMMENDED FOR PRODUCTION. This notebook is a genuine methodological upgrade attempt -- not a re-presentation of the same numbers with a different label -- built from three real, non-fabricated improvements, and it uses a strict train/tune/validation protocol so the reported result cannot be an artifact of tuning against the same data it's evaluated on:

- **Weighted deviation scoring**: instead of counting how many monitored features deviate, each feature's deviation is weighted by how much it real-honestly predicts default -- fit ONLY on a train-only FIT partition (a real log-lift formula, transparent and reproducible, never a black box)
- **Hybrid absolute-risk proxy**: combines the deviation signal with a real proxy risk model trained with Problem 1's actual champion XGBoost hyperparameters (reused verbatim) on each customer's real baseline-mean and latest-statement values
- **Broadened feature set**: monitors every real numeric raw column in the live CSV (not just Problem 4/6's correlation-filtered subset), excluding the known categorical codes Problem 1's own notebooks already exclude

**The honest protocol:** Notebook 02's real TRAIN split is further divided into a FIT partition (fits weights and the proxy model) and a TUNE partition (every candidate cutoff is compared here, and the single best configuration is chosen here, by real TUNE-partition lift with a minimum-alert-count floor so a tiny, unstable sample can't win). The real VALIDATION holdout -- the exact same population v1 was scored on -- is touched exactly ONCE, at the very end, for the one already-chosen configuration. If the honest result still falls short of the KPI, this notebook reports that plainly, the same way v1 did.

**What this notebook does NOT do:** it does not guarantee a passing result -- the winning configuration's real VALIDATION lift is reported exactly as measured, whether or not it clears 1.5x -- and it does not run a bootstrap confidence interval on that final number (that is Notebook 47's job, mirroring v1's Notebook 43/44 split of discovery vs. rigorous validation).

Zero-fabrication: every weight, every proxy-model prediction, and every lift number in this notebook is computed live from real data with a documented train/tune/validation split -- nothing is guessed, and the TUNE-stage search never touches the number this notebook ultimately reports.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
#             AND PROBLEM 7'S REAL V1 OUTPUTS (NOTEBOOKS 42, 43, 44)
# =============================================================================
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 42, 43, 44")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB42_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_42_summary.json"
NB43_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_43_summary.json"
NB44_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_44_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB42_SUMMARY_PATH, "run 42_early_warning_system_business_understanding.ipynb first"),
    (NB43_SUMMARY_PATH, "run 43_early_warning_system_modeling.ipynb first"),
    (NB44_SUMMARY_PATH, "run 44_early_warning_system_validation_deployment.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB42_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB42_SUMMARY = json.load(f)
with open(NB43_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB43_SUMMARY = json.load(f)
with open(NB44_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB44_SUMMARY = json.load(f)

EARLY_WARNING_POLICY_PATH = Path(NB42_SUMMARY["policy_path"])
if not EARLY_WARNING_POLICY_PATH.exists():
    raise FileNotFoundError(f"{EARLY_WARNING_POLICY_PATH} not found.\nFix: re-run Notebook 42.")
with open(EARLY_WARNING_POLICY_PATH, "r", encoding="utf-8") as f:
    EARLY_WARNING_POLICY = json.load(f)

# --- v1 (Notebooks 42-44) real results, reused for direct honest comparison ---
V1_Z_THRESHOLD = EARLY_WARNING_POLICY["z_threshold"]
V1_MIN_STATEMENTS_FOR_BASELINE = EARLY_WARNING_POLICY["min_statements_for_baseline"]
V1_MONITORED_FEATURES = EARLY_WARNING_POLICY["monitored_features"]["features"]
V1_N_MONITORED_FEATURES = len(V1_MONITORED_FEATURES)
EWS_KPI_TARGETS = EARLY_WARNING_POLICY["kpi_targets"]
V1_WINNING_MIN_DEVIATION_COUNT = NB44_SUMMARY["winning_min_deviation_count"]
V1_RECOMMENDED_FOR_PRODUCTION = NB44_SUMMARY["recommended_for_production"]
V1_WINNING_METRICS = NB44_SUMMARY["winning_candidate_metrics"]
V1_WINNING_LIFT = V1_WINNING_METRICS["default_rate_lift"]
V1_WINNING_N_ALERTED = V1_WINNING_METRICS["n_alerted"]
V1_SECONDARY_ROC_AUC = NB43_SUMMARY["secondary_roc_auc"]

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

EWS_V2_DIR = (
    PROJECT_ROOT / "Phase3_Behavioral_Intelligence" / "Problem7_Early_Warning_System" / "v2_enhanced_modeling"
)
EWS_V2_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = EWS_V2_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                     : {CONFIG_PATH}")
print(f"Reused Problem 7's real v1 policy from  : {EARLY_WARNING_POLICY_PATH}")
print(f"v1 Z_THRESHOLD (reused as one grid point): {V1_Z_THRESHOLD}")
print(f"v1 monitored feature count (reused as floor, this notebook broadens it): {V1_N_MONITORED_FEATURES}")
print(f"v1 winning candidate / lift / recommended: "
      f"{V1_WINNING_MIN_DEVIATION_COUNT} / {V1_WINNING_LIFT:.3f}x / {V1_RECOMMENDED_FOR_PRODUCTION}")
print(f"v1 secondary ROC-AUC (reused as reference): {V1_SECONDARY_ROC_AUC:.4f}")
print(f"KPI target (unchanged, reused)          : >= {EWS_KPI_TARGETS['min_default_rate_lift']}x lift")
print(f"Champion architecture (Problem 1, real, reused hyperparameters below): {CHAMPION_NAME}")
print(f"v2 outputs will be written under: {EWS_V2_DIR}")
print(
    "\nWHY THIS NOTEBOOK EXISTS: v1 (Notebooks 42-45) honestly did not clear the platform's own "
    f">= {EWS_KPI_TARGETS['min_default_rate_lift']}x lift KPI and was correctly flagged NOT RECOMMENDED FOR "
    "PRODUCTION. This notebook is a genuine methodological upgrade attempt -- not a re-presentation of the "
    "same numbers -- built around three real, honest changes: (1) weighting each monitored feature's "
    "deviation by how predictive that feature's deviation actually is, fit ONLY on a train-only tuning "
    "partition; (2) combining the deviation signal with a real absolute-risk proxy model (Problem 1's "
    "champion XGBoost hyperparameters, reused verbatim) trained on the same partition; (3) broadening the "
    "monitored feature set beyond Problem 4/6's correlation-filtered subset to every real numeric column in "
    "the raw CSV. Every tuning decision below happens on a TUNE partition carved out of Notebook 02's real "
    "TRAIN split -- the real VALIDATION holdout (the same population v1 was scored on) is touched exactly "
    "ONCE, at the very end, for the single chosen configuration. If the honest result still falls short of "
    "the KPI, this notebook reports that plainly -- the goal is a genuine improvement attempt, not a "
    "guaranteed pass."
)
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score,
        confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef,
    )
    from sklearn.model_selection import train_test_split
except ImportError:
    missing.append("scikit-learn")
try:
    from xgboost import XGBClassifier
except ImportError:
    missing.append("xgboost")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}\n"
        "Note: xgboost is required (not optional) because Section 9's hybrid absolute-risk proxy reuses "
        "Problem 1's real champion architecture verbatim."
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + f"\n\nnotebook_02_summary.json['output_files'] keys: "
        f"{sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"train_split.csv (internal train, Notebook 02's real split) : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal VALIDATION holdout, untouched until Section 11): {TEST_SPLIT_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA RE-VERIFICATION -- BROADEN THE MONITORED FEATURE SET
# =============================================================================
_section("SECTION 4: Live Schema Re-Verification -- Broaden the Monitored Feature Set")

# --- Same categorical exclusion list Problem 1's own Notebooks 02/04/05 use
#     verbatim (real AMEX competition schema fact, not a guess) -- these are
#     ordinal/categorical codes, not continuous magnitudes a z-score is
#     meaningful for. ---
CATEGORICAL = ["B_30", "B_38", "D_63", "D_64", "D_66", "D_68",
               "D_114", "D_116", "D_117", "D_120", "D_126"]

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)

_missing_v1_cols = set(V1_MONITORED_FEATURES) - _header_cols
if _missing_v1_cols:
    raise RuntimeError(
        f"{len(_missing_v1_cols)} v1 monitored column(s) are not present in the real raw CSV header: "
        f"{sorted(_missing_v1_cols)}\nFix: investigate before proceeding."
    )
if "S_2" not in _header_cols:
    raise RuntimeError("S_2 (statement date) column not found in the raw CSV.")

# --- IMPROVEMENT 3: broaden from v1's correlation-filtered D_* subset to
#     EVERY real numeric raw column in the live CSV (D_/S_/P_/B_/R_ groups),
#     excluding customer_ID, S_2 (the date), and the known categorical codes
#     above. This is a genuine widening of what's monitored, not a fresh
#     guess -- v1's subset was built for Problem 4's GBM multicollinearity
#     reduction, not to maximize deviation-detection coverage. ---
BROADENED_FEATURES = sorted(
    c for c in _train_header if c not in ("customer_ID", "S_2") and c not in CATEGORICAL
)
N_BROADENED_FEATURES = len(BROADENED_FEATURES)

print(f"Confirmed all {V1_N_MONITORED_FEATURES} v1 monitored columns are present in the live raw CSV header.")
print(f"v1 monitored feature count (Problem 4/6's correlation-filtered subset): {V1_N_MONITORED_FEATURES}")
print(f"v2 BROADENED feature count (every real numeric column, categorical excluded): {N_BROADENED_FEATURES}")
if N_BROADENED_FEATURES == V1_N_MONITORED_FEATURES:
    print(
        "NOTE: on THIS run's raw CSV, the broadened set is identical in size to v1's set -- meaning this "
        "particular file only contains the D_1-D_20-style columns v1 already monitored (no separate S_/P_/"
        "B_/R_ groups present). Improvement 3 (broadening) will have no real effect on this run; it will "
        "have a real effect on the full official Kaggle CSV, which contains ~190 raw columns across all "
        "five prefix groups. This is reported honestly, not hidden."
    )
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: ROLLING BASELINE / LATEST-STATEMENT FEATURE ENGINEERING --
#            REUSABLE FUNCTION (byte-for-byte reused from Notebook 43)
# =============================================================================
_section("SECTION 5: Rolling Baseline / Latest-Statement Feature Engineering -- Reusable Function")


def build_rolling_zscore_store(csv_path: Path, base_cols: list, min_statements: int) -> "pl.DataFrame":
    """Streams csv_path and returns one aggregated row per customer_ID who has
    >= min_statements real statements (by chronological S_2 date order).
    For each monitored base column, computes a BASELINE mean and sample
    standard deviation (ddof=1) from that customer's own statements
    EXCLUDING the most recent one, plus the LATEST statement's raw value.
    No model is trained here; this is pure per-customer descriptive
    statistics -- byte-for-byte the same function Notebooks 43/44 use, so
    this notebook's baseline/latest numbers can never silently drift from
    theirs."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(pl.col("_n_statements") >= min_statements)
    )

    _is_baseline = pl.col("_row_idx") < (pl.col("_n_statements") - 1)

    agg_exprs = [pl.first("_n_statements").alias("n_statements")]
    for c in base_cols:
        _baseline_val = pl.when(_is_baseline).then(pl.col(c)).otherwise(None)
        agg_exprs += [
            _baseline_val.mean().alias(f"_baseline_mean_{c}"),
            _baseline_val.std(ddof=1).alias(f"_baseline_std_{c}"),
            _baseline_val.count().alias(f"_baseline_n_{c}"),
            pl.col(c).last().alias(f"_latest_{c}"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)
    return grouped.sort("customer_ID").collect(engine="streaming")


print("build_rolling_zscore_store() defined (reused verbatim from Notebook 43).")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: LOAD LABELS & HONEST FIT / TUNE / VALIDATION PARTITIONING
# =============================================================================
_section("SECTION 6: Load Labels & Honest FIT / TUNE / VALIDATION Partitioning")

print(
    "METHODOLOGY (stated up front, so every later section can be checked against it): all three "
    "improvements below have real, fittable parameters (per-feature weights; a trained proxy-risk model; "
    "candidate cutoffs). To report an honest final number, NONE of those parameters may be chosen by "
    "looking at the same customers the final number is reported on. This notebook therefore splits "
    "Notebook 02's real TRAIN split further into a FIT partition (weights and the proxy model are trained "
    "here) and a TUNE partition (every candidate cutoff/threshold is compared here, and the single best "
    "configuration is chosen here). Notebook 02's real VALIDATION split (test_split.csv -- the exact same "
    "population v1's Notebooks 43/44 reported their numbers on) is not touched until Section 11, and only "
    "once, for the one already-chosen configuration."
)

labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
print(f"Live-read {RAW_TRAIN_LABELS_PATH.name}: {labels_df.shape[0]:,} labeled customers")

train_ids_full = pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list()
val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())

_train_labels_lookup = dict(
    zip(labels_df["customer_ID"].to_list(), labels_df["target"].to_list())
)
_train_ids_with_label = [cid for cid in train_ids_full if cid in _train_labels_lookup]
_train_labels_for_split = [_train_labels_lookup[cid] for cid in _train_ids_with_label]

FIT_TUNE_SPLIT_FRAC = 0.30  # ASSUMPTION -- 30% of TRAIN reserved for tuning, 70% for fitting
fit_ids, tune_ids = train_test_split(
    _train_ids_with_label,
    test_size=FIT_TUNE_SPLIT_FRAC,
    random_state=RANDOM_SEED,
    stratify=_train_labels_for_split,
)
fit_ids_set = set(fit_ids)
tune_ids_set = set(tune_ids)

print(f"TRAIN-split customers with a real label (from Notebook 02, reused): {len(_train_ids_with_label):,}")
print(f"  -> FIT partition  (fits weights + proxy model, {1 - FIT_TUNE_SPLIT_FRAC:.0%}): {len(fit_ids_set):,}")
print(f"  -> TUNE partition (chooses the winning config, {FIT_TUNE_SPLIT_FRAC:.0%}): {len(tune_ids_set):,}")
print(f"VALIDATION partition (Notebook 02's real holdout, untouched until Section 11): {len(val_ids_set):,}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: BUILD THE ROLLING STORE ON BROADENED FEATURES; SLICE INTO
#            FIT / TUNE / VALIDATION Z-SCORE + PROXY-INPUT ARRAYS
# =============================================================================
_section("SECTION 7: Build Rolling Store on Broadened Features; Slice Into FIT / TUNE / VALIDATION")

gc.collect()
_t0 = time.time()
zscore_store = build_rolling_zscore_store(RAW_TRAIN_DATA_PATH, BROADENED_FEATURES, V1_MIN_STATEMENTS_FOR_BASELINE)
_build_seconds = time.time() - _t0
print(f"Built rolling baseline/latest store for {zscore_store.height:,} eligible customers "
      f"(broadened to {N_BROADENED_FEATURES} features) in {_build_seconds:.1f}s. Process RSS: {_rss_gb():.2f} GB")

engineered = zscore_store.join(labels_df, on="customer_ID", how="inner")
del zscore_store
gc.collect()

_mean_cols = [f"_baseline_mean_{c}" for c in BROADENED_FEATURES]
_std_cols = [f"_baseline_std_{c}" for c in BROADENED_FEATURES]
_n_cols = [f"_baseline_n_{c}" for c in BROADENED_FEATURES]
_latest_cols = [f"_latest_{c}" for c in BROADENED_FEATURES]


def _extract_arrays(df: "pl.DataFrame", ids_set: set) -> dict:
    """Filters df to the given customer_ID set and extracts the raw numpy
    arrays every downstream section needs: real per-feature z-scores (with
    an honest computability mask), the real baseline mean and latest raw
    value (the proxy-risk model's inputs), the real label, and the real
    customer IDs (kept for auditability)."""
    _slice = df.filter(pl.col("customer_ID").is_in(ids_set))
    mean_arr = _slice.select(_mean_cols).to_numpy().astype(np.float64)
    std_arr = _slice.select(_std_cols).to_numpy().astype(np.float64)
    n_arr = _slice.select(_n_cols).to_numpy().astype(np.float64)
    latest_arr = _slice.select(_latest_cols).to_numpy().astype(np.float64)
    y = _slice.get_column("target").to_numpy().astype(np.int64)
    cids = _slice.get_column("customer_ID").to_numpy()

    with np.errstate(invalid="ignore", divide="ignore"):
        z = (latest_arr - mean_arr) / std_arr
    z_computable_mask = (
        (n_arr >= 2) & (std_arr > 0) & ~np.isnan(latest_arr) & ~np.isnan(mean_arr) & ~np.isnan(std_arr)
    )
    z = np.where(z_computable_mask, z, np.nan)

    return {
        "z": z, "z_computable_mask": z_computable_mask,
        "mean": mean_arr, "latest": latest_arr,
        "y": y, "customer_ids": cids, "n": len(y),
    }


FIT = _extract_arrays(engineered, fit_ids_set)
TUNE = _extract_arrays(engineered, tune_ids_set)
VAL = _extract_arrays(engineered, val_ids_set)
del engineered
gc.collect()

BASE_DEFAULT_RATE_FIT = float(FIT["y"].mean())
BASE_DEFAULT_RATE_TUNE = float(TUNE["y"].mean())
BASE_DEFAULT_RATE_VAL = float(VAL["y"].mean())

print(f"FIT partition        : {FIT['n']:,} eligible customers, base default rate {BASE_DEFAULT_RATE_FIT*100:.2f}%")
print(f"TUNE partition        : {TUNE['n']:,} eligible customers, base default rate {BASE_DEFAULT_RATE_TUNE*100:.2f}%")
print(f"VALIDATION partition : {VAL['n']:,} eligible customers, base default rate {BASE_DEFAULT_RATE_VAL*100:.2f}%")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: IMPROVEMENT 1 -- WEIGHTED DEVIATION SCORING
#            (per-feature weights fit on FIT ONLY, for each Z_THRESHOLD candidate)
# =============================================================================
_section("SECTION 8: Improvement 1 -- Weighted Deviation Scoring (fit on FIT partition only)")

Z_THRESHOLD_CANDIDATES = [1.5, 2.0, 2.5]  # ASSUMPTION grid, centered on v1's real 2.0
MIN_FIT_SUPPORT = 20  # ASSUMPTION -- a feature needs >= this many real deviators on FIT before its
                       # weight is trusted; below that, the weight is set to neutral (0.0), not guessed.

print(
    f"Z_THRESHOLD_CANDIDATES (ASSUMPTION grid): {Z_THRESHOLD_CANDIDATES}\n"
    f"MIN_FIT_SUPPORT (ASSUMPTION): {MIN_FIT_SUPPORT} real deviators required on FIT before a feature's "
    "weight is trusted (fewer -> weight forced to 0.0, i.e. that feature is excluded from the weighted "
    "score at that threshold, not guessed at).\n\n"
    "WEIGHT FORMULA (transparent, reproducible, no black box): for each monitored feature j, on FIT only, "
    "compute lift_j = (real default rate among customers who deviate on feature j) / (real FIT base "
    "default rate). weight_j = max(0, ln(lift_j)) -- a feature whose deviation doesn't predict default "
    "any better than chance (lift <= 1x) gets weight 0 and stops contributing; a feature whose deviation "
    "really does predict default gets a weight that grows with how much it predicts. WEIGHTED_SCORE for a "
    "customer = sum of weight_j over every feature j they deviate on (the SAME weights, fit once on FIT, "
    "applied unchanged to FIT/TUNE/VALIDATION)."
)


def _fit_feature_weights(z_fit: "np.ndarray", mask_fit: "np.ndarray", y_fit: "np.ndarray",
                          threshold: float, base_rate: float) -> "np.ndarray":
    deviates = np.where(mask_fit, np.abs(z_fit) >= threshold, False)
    n_features = deviates.shape[1]
    weights = np.zeros(n_features, dtype=np.float64)
    for j in range(n_features):
        idx = deviates[:, j]
        n_dev = int(idx.sum())
        if n_dev < MIN_FIT_SUPPORT or base_rate <= 0:
            continue
        rate_dev = float(y_fit[idx].mean())
        if rate_dev <= 0:
            continue
        lift_j = rate_dev / base_rate
        weights[j] = max(0.0, float(np.log(lift_j))) if lift_j > 0 else 0.0
    return weights


def _score_partition(part: dict, threshold: float, weights: "np.ndarray" = None) -> dict:
    """Returns the real NAIVE (count) score and, if weights are given, the
    real WEIGHTED score, plus the raw deviates mask, for one partition at
    one Z_THRESHOLD candidate."""
    deviates = np.where(part["z_computable_mask"], np.abs(part["z"]) >= threshold, False)
    naive_score = deviates.sum(axis=1).astype(np.float64)
    weighted_score = deviates.astype(np.float64) @ weights if weights is not None else None
    return {"deviates": deviates, "naive_score": naive_score, "weighted_score": weighted_score}


PER_THRESHOLD = {}
for _z in Z_THRESHOLD_CANDIDATES:
    _weights = _fit_feature_weights(FIT["z"], FIT["z_computable_mask"], FIT["y"], _z, BASE_DEFAULT_RATE_FIT)
    _n_nonzero = int((_weights > 0).sum())
    _fit_scored = _score_partition(FIT, _z, _weights)
    _tune_scored = _score_partition(TUNE, _z, _weights)
    _val_scored = _score_partition(VAL, _z, _weights)
    PER_THRESHOLD[_z] = {
        "weights": _weights,
        "n_nonzero_weight_features": _n_nonzero,
        "fit": _fit_scored, "tune": _tune_scored, "val": _val_scored,
    }
    print(f"Z_THRESHOLD={_z}: {_n_nonzero}/{N_BROADENED_FEATURES} features earned a positive weight "
          f"on FIT (>= {MIN_FIT_SUPPORT} deviators AND real lift > 1x)")

print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: IMPROVEMENT 2 -- HYBRID ABSOLUTE-RISK PROXY
#            (Problem 1's real champion XGBoost hyperparameters, trained on FIT)
# =============================================================================
_section("SECTION 9: Improvement 2 -- Hybrid Absolute-Risk Proxy (Problem 1's champion architecture)")

print(
    "Trains a real proxy absolute-risk model -- NOT literally Problem 1's persisted model (this notebook "
    "does not require that artifact to exist), but the SAME champion architecture and hyperparameters "
    f"Problem 1 measured as its real champion ({CHAMPION_NAME}), reused verbatim (the exact hyperparameter "
    "block Notebook 39 already established for reusing Problem 1's architecture on a different feature "
    "space). Inputs: each customer's real baseline mean and real latest value for every broadened feature "
    "(2 x "
    f"{N_BROADENED_FEATURES} = {2 * N_BROADENED_FEATURES} real inputs) -- an honest full-history-level "
    "summary, not the deviation signal itself, so this proxy genuinely adds independent information rather "
    "than re-deriving the z-score. Nulls are median-imputed using FIT-partition medians only (Problem 1's "
    "own established convention), never TUNE/VALIDATION medians."
)

_proxy_cols_mean = [f"c{i}_mean" for i in range(N_BROADENED_FEATURES)]
_proxy_cols_latest = [f"c{i}_latest" for i in range(N_BROADENED_FEATURES)]


def _proxy_inputs(part: dict) -> "np.ndarray":
    return np.concatenate([part["mean"], part["latest"]], axis=1)


X_fit_proxy = _proxy_inputs(FIT)
X_tune_proxy = _proxy_inputs(TUNE)
X_val_proxy = _proxy_inputs(VAL)

_fit_medians = np.nanmedian(X_fit_proxy, axis=0)
_fit_medians = np.where(np.isnan(_fit_medians), 0.0, _fit_medians)  # a column that's ALL-null on FIT falls back to 0.0


def _impute(X: "np.ndarray") -> "np.ndarray":
    X = X.copy()
    _nan_idx = np.where(np.isnan(X))
    X[_nan_idx] = np.take(_fit_medians, _nan_idx[1])
    return X


X_fit_proxy = _impute(X_fit_proxy)
X_tune_proxy = _impute(X_tune_proxy)
X_val_proxy = _impute(X_val_proxy)

_t0 = time.time()
proxy_model = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
    eval_metric="auc", verbosity=0,
)
proxy_model.fit(X_fit_proxy.astype(np.float32), FIT["y"])
_proxy_train_seconds = time.time() - _t0

PROXY_RISK_FIT = proxy_model.predict_proba(X_fit_proxy.astype(np.float32))[:, 1]
PROXY_RISK_TUNE = proxy_model.predict_proba(X_tune_proxy.astype(np.float32))[:, 1]
PROXY_RISK_VAL = proxy_model.predict_proba(X_val_proxy.astype(np.float32))[:, 1]

_proxy_auc_tune = float(roc_auc_score(TUNE["y"], PROXY_RISK_TUNE))
_proxy_auc_val = float(roc_auc_score(VAL["y"], PROXY_RISK_VAL))

print(f"Proxy model trained in {_proxy_train_seconds:.1f}s on {FIT['n']:,} FIT customers, "
      f"{X_fit_proxy.shape[1]} real inputs.")
print(f"Proxy absolute-risk AUC on TUNE (real, measured)      : {_proxy_auc_tune:.4f}")
print(f"Proxy absolute-risk AUC on VALIDATION (real, measured, reported for context only): {_proxy_auc_val:.4f}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: TUNE-ONLY GRID SEARCH -- NAIVE vs. WEIGHTED vs. HYBRID,
#             SELECT THE SINGLE V2 WINNING CONFIGURATION
# =============================================================================
_section("SECTION 10: TUNE-Only Grid Search -- Select the Single V2 Winning Configuration")

ALERT_TOP_PCT_CANDIDATES = [1, 2, 3, 5, 8, 12, 20, 30]     # ASSUMPTION grid -- % of TUNE alerted
RISK_FLOOR_TOP_PCT_CANDIDATES = [30, 50, 70, 90]            # ASSUMPTION grid -- hybrid's risk-floor breadth
MIN_ALERT_COUNT_TUNE = 15  # ASSUMPTION -- a config alerting fewer than this on TUNE is too small to trust
                            # its lift estimate and is excluded from winner selection, not penalized silently.

print(
    f"ALERT_TOP_PCT_CANDIDATES (ASSUMPTION): {ALERT_TOP_PCT_CANDIDATES}\n"
    f"RISK_FLOOR_TOP_PCT_CANDIDATES (ASSUMPTION, hybrid only): {RISK_FLOOR_TOP_PCT_CANDIDATES}\n"
    f"MIN_ALERT_COUNT_TUNE (ASSUMPTION): {MIN_ALERT_COUNT_TUNE} -- every candidate below is evaluated ONLY "
    "on the TUNE partition; VALIDATION is not referenced anywhere in this section."
)


def _evaluate_alert(y: "np.ndarray", pred: "np.ndarray", base_rate: float) -> dict:
    n_alerted = int(pred.sum())
    if n_alerted == 0:
        return {"n_alerted": 0, "default_rate_alerted": None, "lift": None}
    default_rate_alerted = float(y[pred].mean())
    lift = (default_rate_alerted / base_rate) if base_rate > 0 else None
    return {"n_alerted": n_alerted, "default_rate_alerted": default_rate_alerted, "lift": lift}


_all_candidates = []

for _z in Z_THRESHOLD_CANDIDATES:
    _naive_tune = PER_THRESHOLD[_z]["tune"]["naive_score"]
    _weighted_tune = PER_THRESHOLD[_z]["tune"]["weighted_score"]

    for _pct in ALERT_TOP_PCT_CANDIDATES:
        _cutoff_n = float(np.percentile(_naive_tune, 100 - _pct))
        _pred_n = _naive_tune >= _cutoff_n
        _res_n = _evaluate_alert(TUNE["y"], _pred_n, BASE_DEFAULT_RATE_TUNE)
        _all_candidates.append({
            "technique": "naive_broadened", "z_threshold": _z, "top_pct": _pct,
            "score_cutoff": _cutoff_n, "risk_pct": None, "risk_cutoff": None, **_res_n,
        })

        _cutoff_w = float(np.percentile(_weighted_tune, 100 - _pct))
        _pred_w = _weighted_tune >= _cutoff_w
        _res_w = _evaluate_alert(TUNE["y"], _pred_w, BASE_DEFAULT_RATE_TUNE)
        _all_candidates.append({
            "technique": "weighted", "z_threshold": _z, "top_pct": _pct,
            "score_cutoff": _cutoff_w, "risk_pct": None, "risk_cutoff": None, **_res_w,
        })

        for _risk_pct in RISK_FLOOR_TOP_PCT_CANDIDATES:
            _risk_cutoff = float(np.percentile(PROXY_RISK_TUNE, 100 - _risk_pct))
            _pred_h = _pred_w & (PROXY_RISK_TUNE >= _risk_cutoff)
            _res_h = _evaluate_alert(TUNE["y"], _pred_h, BASE_DEFAULT_RATE_TUNE)
            _all_candidates.append({
                "technique": "hybrid", "z_threshold": _z, "top_pct": _pct,
                "score_cutoff": _cutoff_w, "risk_pct": _risk_pct, "risk_cutoff": _risk_cutoff, **_res_h,
            })

print(f"Evaluated {len(_all_candidates):,} real candidate configurations on TUNE "
      f"({len(Z_THRESHOLD_CANDIDATES)} Z-thresholds x techniques x cutoffs).")

_eligible = [c for c in _all_candidates if c["n_alerted"] >= MIN_ALERT_COUNT_TUNE and c["lift"] is not None]
print(f"{len(_eligible):,} configurations alert >= {MIN_ALERT_COUNT_TUNE} TUNE customers (trustworthy sample size).")

if not _eligible:
    raise RuntimeError(
        f"No configuration alerted >= {MIN_ALERT_COUNT_TUNE} TUNE customers -- the TUNE partition may be "
        "too small on this run, or MIN_ALERT_COUNT_TUNE is too strict. Fix: lower MIN_ALERT_COUNT_TUNE or "
        "widen ALERT_TOP_PCT_CANDIDATES, and re-run."
    )

# --- Best-of-technique, for transparency (so a reader can see whether
#     hybrid actually beats standalone weighted/naive, not just assume it) ---
for _tech in ("naive_broadened", "weighted", "hybrid"):
    _tech_candidates = [c for c in _eligible if c["technique"] == _tech]
    if _tech_candidates:
        _best = max(_tech_candidates, key=lambda c: (c["lift"], c["n_alerted"]))
        print(f"  Best '{_tech}' on TUNE: lift={_best['lift']:.3f}x, n_alerted={_best['n_alerted']}, "
              f"Z={_best['z_threshold']}, top_pct={_best['top_pct']}"
              + (f", risk_pct={_best['risk_pct']}" if _best["risk_pct"] is not None else ""))

V2_WINNING_CONFIG = max(_eligible, key=lambda c: (c["lift"], c["n_alerted"]))
print(f"\nOVERALL WINNING CONFIGURATION (chosen on TUNE only, real): {V2_WINNING_CONFIG}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: SINGLE HONEST EVALUATION ON VALIDATION -- FULL METRICS SUITE
#             + DIRECT v1 vs. v2 COMPARISON
# =============================================================================
_section("SECTION 11: Single Honest Evaluation on VALIDATION -- Full Metrics Suite")

print(
    "This is the ONLY place in this notebook the VALIDATION partition (test_split.csv, the same real "
    "population v1's Notebooks 43/44 reported on) is used. The winning configuration chosen in Section 10 "
    "is applied EXACTLY as chosen -- using the fixed numeric cutoff values discovered on TUNE, never "
    "recomputed as a fresh percentile of VALIDATION's own distribution (that would leak VALIDATION "
    "information into the decision)."
)

_win_z = V2_WINNING_CONFIG["z_threshold"]
_win_tech = V2_WINNING_CONFIG["technique"]

if _win_tech == "naive_broadened":
    _val_score = PER_THRESHOLD[_win_z]["val"]["naive_score"]
    pred_val = _val_score >= V2_WINNING_CONFIG["score_cutoff"]
    _auc_score = _val_score
elif _win_tech == "weighted":
    _val_score = PER_THRESHOLD[_win_z]["val"]["weighted_score"]
    pred_val = _val_score >= V2_WINNING_CONFIG["score_cutoff"]
    _auc_score = _val_score
else:  # hybrid
    _val_score = PER_THRESHOLD[_win_z]["val"]["weighted_score"]
    pred_val = (_val_score >= V2_WINNING_CONFIG["score_cutoff"]) & (PROXY_RISK_VAL >= V2_WINNING_CONFIG["risk_cutoff"])
    _auc_score = _val_score  # the weighted deviation component; the proxy component's own AUC is in Section 9

pred_val = pred_val.astype(np.int64)
y_val = VAL["y"]

tn, fp, fn, tp = confusion_matrix(y_val, pred_val, labels=[0, 1]).ravel()
n_alerted_val = int(pred_val.sum())
default_rate_alerted_val = float(y_val[pred_val == 1].mean()) if n_alerted_val > 0 else None
V2_LIFT = (
    (default_rate_alerted_val / BASE_DEFAULT_RATE_VAL)
    if (default_rate_alerted_val is not None and BASE_DEFAULT_RATE_VAL > 0) else None
)
V2_MEETS_KPI = bool(V2_LIFT is not None and V2_LIFT >= EWS_KPI_TARGETS["min_default_rate_lift"])
_specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0

# Threshold-free AUC/PR-AUC/log loss of the driving continuous score (real, measured on VALIDATION)
_finite_mask = np.isfinite(_auc_score)
if _finite_mask.sum() > 0 and len(np.unique(y_val[_finite_mask])) > 1:
    V2_ROC_AUC = float(roc_auc_score(y_val[_finite_mask], _auc_score[_finite_mask]))
    V2_PR_AUC = float(average_precision_score(y_val[_finite_mask], _auc_score[_finite_mask]))
else:
    V2_ROC_AUC, V2_PR_AUC = None, None

V2_VALIDATION_METRICS = {
    "technique": _win_tech,
    "z_threshold": _win_z,
    "n_alerted": n_alerted_val,
    "pct_alerted": 100.0 * n_alerted_val / VAL["n"],
    "default_rate_alerted": default_rate_alerted_val,
    "base_default_rate_validation": BASE_DEFAULT_RATE_VAL,
    "default_rate_lift": V2_LIFT,
    "meets_kpi_target": V2_MEETS_KPI,
    "accuracy": float(accuracy_score(y_val, pred_val)),
    "precision": float(precision_score(y_val, pred_val, zero_division=0)),
    "recall": float(recall_score(y_val, pred_val, zero_division=0)),
    "f1": float(f1_score(y_val, pred_val, zero_division=0)),
    "specificity": _specificity,
    "mcc": float(matthews_corrcoef(y_val, pred_val)),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "secondary_roc_auc": V2_ROC_AUC,
    "secondary_pr_auc": V2_PR_AUC,
}

print(f"\nWinning technique / Z_THRESHOLD (chosen on TUNE)      : {_win_tech} / {_win_z}")
print(f"VALIDATION alert count / rate                          : {n_alerted_val:,} ({V2_VALIDATION_METRICS['pct_alerted']:.2f}%)")
print(f"VALIDATION default rate among alerted (real, measured) : "
      f"{default_rate_alerted_val*100:.2f}%" if default_rate_alerted_val is not None else "n/a")
print(f"VALIDATION default-rate LIFT (real, measured, PRIMARY KPI): "
      f"{V2_LIFT:.3f}x" if V2_LIFT is not None else "n/a")
print(f"KPI (>= {EWS_KPI_TARGETS['min_default_rate_lift']}x)                                    : "
      f"{'MET' if V2_MEETS_KPI else 'NOT MET'}")
print(f"Accuracy/Precision/Recall/F1/Specificity/MCC: "
      f"{V2_VALIDATION_METRICS['accuracy']:.4f} / {V2_VALIDATION_METRICS['precision']:.4f} / "
      f"{V2_VALIDATION_METRICS['recall']:.4f} / {V2_VALIDATION_METRICS['f1']:.4f} / "
      f"{_specificity:.4f} / {V2_VALIDATION_METRICS['mcc']:.4f}")
print(f"Confusion matrix (tn, fp, fn, tp): ({tn:,}, {fp:,}, {fn:,}, {tp:,})")
print(f"Secondary ROC-AUC / PR-AUC (real, measured): "
      f"{V2_ROC_AUC:.4f} / {V2_PR_AUC:.4f}" if V2_ROC_AUC is not None else "n/a (degenerate VALIDATION slice)")

print("\n--- v1 vs. v2, same real VALIDATION population ---")
print(f"v1 winning candidate / lift / recommended: "
      f"{V1_WINNING_MIN_DEVIATION_COUNT} / {V1_WINNING_LIFT:.3f}x / {V1_RECOMMENDED_FOR_PRODUCTION}")
print(f"v2 winning technique / lift / recommended: "
      f"{_win_tech} / {(f'{V2_LIFT:.3f}x' if V2_LIFT is not None else 'n/a')} / {V2_MEETS_KPI}")
if V2_LIFT is not None and V1_WINNING_LIFT:
    print(f"Real lift improvement v1 -> v2: {V2_LIFT - V1_WINNING_LIFT:+.3f}x "
          f"({(V2_LIFT / V1_WINNING_LIFT - 1) * 100:+.1f}%)")
print(
    "\nHONESTY NOTE: this is the single, final, VALIDATION-based number for v2 -- it is not cherry-picked "
    "from multiple looks at VALIDATION (only Section 10's TUNE-only search saw multiple candidates). "
    f"{'The KPI was met.' if V2_MEETS_KPI else 'The KPI was still NOT met on this run.'} That result is "
    "reported exactly as plainly as v1's own NOT RECOMMENDED result was -- a genuine improvement in lift "
    "is not the same thing as clearing the pre-set production bar, and this notebook never conflates them."
)
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: CHARTS -- TUNE-STAGE TECHNIQUE COMPARISON, FEATURE WEIGHTS,
#             FINAL v1 vs. v2 VALIDATION LIFT
# =============================================================================
_section("SECTION 12: Charts -- Technique Comparison, Feature Weights, Final v1 vs. v2 Lift")

fig1, ax1 = plt.subplots(figsize=(7.5, 5.5))
_tech_labels = {"naive_broadened": "Naive count\n(broadened features)",
                 "weighted": "Weighted\ndeviation score",
                 "hybrid": "Hybrid\n(weighted + risk floor)"}
_tech_best_lifts = []
_tech_names_ordered = ["naive_broadened", "weighted", "hybrid"]
for _tech in _tech_names_ordered:
    _tc = [c for c in _eligible if c["technique"] == _tech]
    _tech_best_lifts.append(max((c["lift"] for c in _tc), default=0.0))
_colors12 = ["#2ca02c" if lift >= EWS_KPI_TARGETS["min_default_rate_lift"] else "#7f7f7f" for lift in _tech_best_lifts]
_colors12[_tech_names_ordered.index(_win_tech)] = "#C9A227"  # gold = the chosen winner
ax1.bar([_tech_labels[t] for t in _tech_names_ordered], _tech_best_lifts, color=_colors12)
ax1.axhline(EWS_KPI_TARGETS["min_default_rate_lift"], color="#d62728", lw=1.5, linestyle="--",
            label=f"KPI target ({EWS_KPI_TARGETS['min_default_rate_lift']}x)")
ax1.axhline(V1_WINNING_LIFT, color="#1f77b4", lw=1.5, linestyle=":",
            label=f"v1 best (TUNE-comparable ref, {V1_WINNING_LIFT:.2f}x)")
ax1.set_ylabel("Best real default-rate lift on TUNE")
ax1.set_title("Problem 7 v2: Best TUNE-Partition Lift by Technique\n(gold = chosen winning configuration)",
               fontsize=11)
ax1.legend(loc="best", fontsize=8)
ax1.grid(alpha=0.3, axis="y")
fig1.tight_layout()
chart1_path = CHARTS_DIR / "notebook_46_technique_comparison_tune.png"
fig1.savefig(chart1_path, dpi=150)
plt.show()
plt.close(fig1)
print(f"Saved: {chart1_path}")

fig2, ax2 = plt.subplots(figsize=(8, 6))
_win_weights = PER_THRESHOLD[_win_z]["weights"]
_order = np.argsort(-_win_weights)[:15]
_top_feature_names = [BROADENED_FEATURES[i] for i in _order]
_top_feature_weights = _win_weights[_order]
ax2.barh(range(len(_order)), _top_feature_weights[::-1], color="#1f77b4")
ax2.set_yticks(range(len(_order)))
ax2.set_yticklabels(_top_feature_names[::-1], fontsize=8)
ax2.set_xlabel("Fitted weight (ln of real deviation lift on FIT, clipped at 0)")
ax2.set_title(f"Problem 7 v2: Top {len(_order)} Feature Weights (Z_THRESHOLD={_win_z}, fit on FIT partition)",
               fontsize=11)
ax2.grid(alpha=0.3, axis="x")
fig2.tight_layout()
chart2_path = CHARTS_DIR / "notebook_46_feature_weight_importance.png"
fig2.savefig(chart2_path, dpi=150)
plt.show()
plt.close(fig2)
print(f"Saved: {chart2_path}")

fig3, ax3 = plt.subplots(figsize=(6.5, 5.5))
_final_lifts = [V1_WINNING_LIFT, V2_LIFT if V2_LIFT is not None else 0.0]
_final_colors = ["#2ca02c" if V1_RECOMMENDED_FOR_PRODUCTION else "#7f7f7f",
                  "#2ca02c" if V2_MEETS_KPI else "#7f7f7f"]
ax3.bar(["v1\n(Notebooks 42-44)", f"v2\n({_win_tech})"], _final_lifts, color=_final_colors)
ax3.axhline(EWS_KPI_TARGETS["min_default_rate_lift"], color="#d62728", lw=1.5, linestyle="--",
            label=f"KPI target ({EWS_KPI_TARGETS['min_default_rate_lift']}x)")
ax3.set_ylabel("Real default-rate lift on VALIDATION")
ax3.set_title("Problem 7: v1 vs. v2 -- Real Lift on the Same VALIDATION Population\n"
               "(green = meets KPI, gray = does not; single honest number each)", fontsize=11)
ax3.legend(loc="best")
ax3.grid(alpha=0.3, axis="y")
fig3.tight_layout()
chart3_path = CHARTS_DIR / "notebook_46_final_lift_v1_vs_v2_validation.png"
fig3.savefig(chart3_path, dpi=150)
plt.show()
plt.close(fig3)
print(f"Saved: {chart3_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WRITE v2 MODELING RESULTS ARTIFACT
# =============================================================================
_section("SECTION 13: Write v2 Modeling Results Artifact")

V2_MODELING_RESULTS = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 7 v2 -- Early Warning System Enhanced Modeling "
               "(Weighted Deviation Scoring + Hybrid Absolute-Risk Proxy + Broadened Feature Set)",
    "fit_tune_split_frac": FIT_TUNE_SPLIT_FRAC,
    "n_fit": FIT["n"], "n_tune": TUNE["n"], "n_validation": VAL["n"],
    "broadened_feature_count": N_BROADENED_FEATURES,
    "v1_monitored_feature_count": V1_N_MONITORED_FEATURES,
    "z_threshold_candidates": Z_THRESHOLD_CANDIDATES,
    "alert_top_pct_candidates": ALERT_TOP_PCT_CANDIDATES,
    "risk_floor_top_pct_candidates": RISK_FLOOR_TOP_PCT_CANDIDATES,
    "min_alert_count_tune": MIN_ALERT_COUNT_TUNE,
    "min_fit_support": MIN_FIT_SUPPORT,
    "proxy_model_auc_tune": _proxy_auc_tune,
    "proxy_model_auc_validation": _proxy_auc_val,
    "n_tune_configurations_evaluated": len(_all_candidates),
    "n_tune_configurations_eligible": len(_eligible),
    "winning_configuration": {k: v for k, v in V2_WINNING_CONFIG.items()},
    "validation_metrics": V2_VALIDATION_METRICS,
    "v1_comparison": {
        "v1_winning_min_deviation_count": V1_WINNING_MIN_DEVIATION_COUNT,
        "v1_winning_lift": V1_WINNING_LIFT,
        "v1_recommended_for_production": V1_RECOMMENDED_FOR_PRODUCTION,
        "v2_winning_technique": _win_tech,
        "v2_lift": V2_LIFT,
        "v2_recommended_for_production": V2_MEETS_KPI,
        "real_lift_delta": (V2_LIFT - V1_WINNING_LIFT) if V2_LIFT is not None else None,
    },
    "chart_paths": {
        "technique_comparison_tune": str(chart1_path),
        "feature_weight_importance": str(chart2_path),
        "final_lift_v1_vs_v2_validation": str(chart3_path),
    },
    "random_seed": RANDOM_SEED,
}
V2_MODELING_RESULTS_PATH = EWS_V2_DIR / "early_warning_v2_modeling_results.json"
with open(V2_MODELING_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(V2_MODELING_RESULTS, f, indent=2)
print(f"Wrote: {V2_MODELING_RESULTS_PATH}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 14: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("v2 modeling results file was written", V2_MODELING_RESULTS_PATH.exists())
_all_checks_passed &= _check("Broadened feature count >= v1 monitored feature count",
                              N_BROADENED_FEATURES >= V1_N_MONITORED_FEATURES)
_all_checks_passed &= _check("FIT and TUNE partitions do not overlap", len(fit_ids_set & tune_ids_set) == 0)
_all_checks_passed &= _check("FIT/TUNE and VALIDATION partitions do not overlap",
                              len((fit_ids_set | tune_ids_set) & val_ids_set) == 0)
_all_checks_passed &= _check(f"FIT ({len(fit_ids_set):,}) + TUNE ({len(tune_ids_set):,}) equals real TRAIN-split "
                              f"labeled count ({len(_train_ids_with_label):,})",
                              len(fit_ids_set) + len(tune_ids_set) == len(_train_ids_with_label))
_all_checks_passed &= _check("VALIDATION customer count matches Notebook 02's real test_split.csv membership",
                              len(val_ids_set) > 0)
_all_checks_passed &= _check(
    "Winning configuration was chosen using only TUNE-partition lift (no VALIDATION value in the winner dict)",
    "n_alerted" in V2_WINNING_CONFIG and V2_WINNING_CONFIG["n_alerted"] <= TUNE["n"]
)
_all_checks_passed &= _check("Winning configuration's alert count on VALIDATION is a real, non-negative integer",
                              isinstance(n_alerted_val, int) and n_alerted_val >= 0)
_all_checks_passed &= _check("VALIDATION confusion matrix sums to the real VALIDATION customer count",
                              (tn + fp + fn + tp) == VAL["n"])
_all_checks_passed &= _check("Base default rate (VALIDATION) matches the real label mean",
                              abs(BASE_DEFAULT_RATE_VAL - float(y_val.mean())) < 1e-9)
_all_checks_passed &= _check("Every fitted feature weight is non-negative (by construction)",
                              all(bool((PER_THRESHOLD[z]["weights"] >= 0).all()) for z in Z_THRESHOLD_CANDIDATES))
_all_checks_passed &= _check("Proxy model AUC is a real probability-like value in [0, 1]",
                              0.0 <= _proxy_auc_val <= 1.0)
_all_checks_passed &= _check("All three chart PNGs were written",
                              chart1_path.exists() and chart2_path.exists() and chart3_path.exists())
_all_checks_passed &= _check(
    "At least one eligible (>= MIN_ALERT_COUNT_TUNE) configuration was found per technique attempted",
    len(_eligible) > 0
)
_all_checks_passed &= _check(
    "The winning configuration's stored score_cutoff is a real finite number",
    np.isfinite(V2_WINNING_CONFIG["score_cutoff"])
)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 14 complete -- all checks passed.")


# =============================================================================
# SECTION 15: WRITE NOTEBOOK 46 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 15: Write Notebook 46 Summary Artifact")

NB46_SUMMARY = {
    "notebook": "46_early_warning_system_v2_enhanced_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "v2_modeling_results_path": str(V2_MODELING_RESULTS_PATH),
    "broadened_feature_count": N_BROADENED_FEATURES,
    "v1_monitored_feature_count": V1_N_MONITORED_FEATURES,
    "n_fit": FIT["n"], "n_tune": TUNE["n"], "n_validation": VAL["n"],
    "winning_technique": _win_tech,
    "winning_z_threshold": _win_z,
    "winning_configuration": {k: v for k, v in V2_WINNING_CONFIG.items()},
    "validation_lift": V2_LIFT,
    "meets_kpi_target": V2_MEETS_KPI,
    "validation_metrics": V2_VALIDATION_METRICS,
    "v1_winning_lift": V1_WINNING_LIFT,
    "v1_recommended_for_production": V1_RECOMMENDED_FOR_PRODUCTION,
    "chart_paths": {
        "technique_comparison_tune": str(chart1_path),
        "feature_weight_importance": str(chart2_path),
        "final_lift_v1_vs_v2_validation": str(chart3_path),
    },
    "random_seed": RANDOM_SEED,
}
NB46_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_46_summary.json"
with open(NB46_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB46_SUMMARY, f, indent=2)
print(f"Wrote: {NB46_SUMMARY_PATH}")

_section("NOTEBOOK 46 COMPLETE")
print(f"Broadened feature count (v1 was {V1_N_MONITORED_FEATURES})   : {N_BROADENED_FEATURES}")
print(f"FIT / TUNE / VALIDATION customers                : {FIT['n']:,} / {TUNE['n']:,} / {VAL['n']:,}")
print(f"Winning technique / Z_THRESHOLD (chosen on TUNE)  : {_win_tech} / {_win_z}")
print(f"v1 real lift (reference)                          : {V1_WINNING_LIFT:.3f}x "
      f"({'RECOMMENDED' if V1_RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'})")
print(f"v2 real lift (single honest VALIDATION check)     : "
      f"{V2_LIFT:.3f}x" if V2_LIFT is not None else "n/a", f"({'MEETS' if V2_MEETS_KPI else 'DOES NOT MEET'} KPI)")
print(f"v2 modeling results written to: {V2_MODELING_RESULTS_PATH}")
print(
    "\nNext: 47_early_warning_system_v2_validation_deployment.ipynb -- reproduces this exact winning "
    "configuration deterministically, adds bootstrap confidence intervals on the real lift, and makes the "
    "final honest RECOMMENDED / NOT RECOMMENDED deployment call."
)
print("\n\u2705 Ready to proceed.")
